# Project Implementation Notes & Requirements

## 📋 Project Roadmap

### 1. Expand Nutrient Coverage
**Goal:** Include both macro and micro nutrients for fast food restaurant items

**Tasks:**
- [ ] Add micro-nutrients (Vitamins, Minerals) in addition to current macros
- [ ] Current macros: Calories, Protein, Carbs, Fat
- [ ] Add micro-nutrients: Vitamin A, Vitamin C, Calcium, Iron, Sodium, Fiber, etc.
- [ ] Update `nutrients` set to include all new nutrients
- [ ] Update `nutrient_per_serving` parameter with complete nutrient data
- [ ] Update `Nmin` and `Nmax` constraints for all nutrients

---

### 2. Fast Food Restaurant Data Collection
**Goal:** Replace generic foods with specific fast food restaurant menu items

**Tasks:**
- [ ] Replace current food items with restaurant-specific items:
  - Chipotle menu items (burrito, bowl, tacos, etc.)
  - Subway menu items (sandwiches, salads, etc.)
  - Pizza Hut menu items (pizzas, sides, etc.)
  - Add more restaurants as needed
- [ ] Manually collect data from restaurant websites/nutrition databases
- [ ] For each menu item, collect:
  - Price per item
  - All macro and micro nutrients
  - Satisfaction score (user preference)
- [ ] Update `foods` set with restaurant menu items
- [ ] Update all parameters (price, nutrients, satisfaction) with real data

---

### 3. Enhanced Objective Function with Uncertainty
**Goal:** Add uncertainty/stochasticity to objective function and constraints

**Current Objective:** Minimize cost while maximizing satisfaction (satisfaction as constraint)

**Tasks:**
- [ ] Add uncertainty to **satisfaction**:
  - Make satisfaction a function of time/context (e.g., satisfaction varies by meal time)
  - Or make satisfaction stochastic (satisfaction scenarios)
  - Or make satisfaction depend on other variables (e.g., satisfaction = f(price, nutrients))
- [ ] Add uncertainty to **prices**:
  - Price scenarios (Low/Medium/High)
  - Price as function of time/season
  - Expected cost minimization
- [ ] Add uncertainty to **nutrients**:
  - Nutrient content variation (actual values vary ±10-15%)
  - Chance constraints: P(nutrient >= min) >= 0.9
- [ ] Update objective to handle uncertainty:
  - Expected cost + risk measure (e.g., CVaR)
  - Multi-objective: cost vs. satisfaction vs. risk
- [ ] Update constraints to handle uncertainty:
  - Stochastic constraints
  - Robust optimization (worst-case)
  - Chance constraints

**Mathematical Formulation Ideas:**
- Satisfaction as function: `satisfaction[i] = base_satisfaction[i] * f(price[i], time, context)`
- Price uncertainty: `E[cost] = Σ scenarios P(s) * cost(s)`
- Nutrient uncertainty: `P(Σ nutrients >= min) >= α`

---

### 4. Post-Optimization Analysis

#### 4.1 Price Sensitivity Analysis
**Tasks:**
- [ ] Extract shadow prices/dual values for cost constraints
- [ ] Identify which foods are most cost-sensitive (highest shadow prices)
- [ ] Calculate price break-even points:
  - How much can each food's price change before solution changes?
  - Price elasticity analysis
- [ ] What-if scenario analysis:
  - "What if Chipotle price increases by 20%?" → Re-solve and compare
  - "What if all restaurant prices increase by 10%?" → Impact analysis
  - Create sensitivity table/matrix

#### 4.2 Nutrient Sensitivity Analysis
**Tasks:**
- [ ] Extract nutrient shadow prices (dual values for nutrient constraints)
- [ ] Calculate value of relaxing each nutrient constraint:
  - "If I need 10 more grams of protein, how much does cost increase?"
  - "What's the cost of increasing calories by 100?"
- [ ] Identify binding constraints:
  - Which nutrient limits are actually constraining the solution?
  - Which nutrients have slack (not binding)?
- [ ] Nutrient trade-off analysis:
  - Show cost impact of changing each nutrient requirement
  - Create nutrient sensitivity report

#### 4.3 Satisfaction Sensitivity Analysis
**Tasks:**
- [ ] Vary `min_satisfaction` threshold and observe solution changes:
  - Solve for min_satisfaction = 200, 250, 300, 350, 400
  - Plot cost vs. satisfaction threshold
- [ ] Analyze food satisfaction impact:
  - Which foods contribute most to total satisfaction?
  - Satisfaction efficiency: satisfaction per dollar
- [ ] Satisfaction elasticity:
  - How does solution change with satisfaction preferences?

---

### 5. Visualization & Analysis

#### 5.1 Nutrient Analysis Visualizations
**Tasks:**
- [ ] Stacked bar chart: Nutrient contributions by food
- [ ] Nutrient balance radar/spider chart: All nutrients in one view
- [ ] Nutrient efficiency chart: Nutrients per dollar spent
- [ ] Nutrient gap analysis: Which nutrients are just meeting minimums?
- [ ] Nutrient distribution: Pie/bar charts showing nutrient breakdown

#### 5.2 Cost Analysis Visualizations
**Tasks:**
- [ ] Cost breakdown pie chart: Cost by food category (Main/Drink/Side)
- [ ] Cost per serving bar chart: Compare prices across foods
- [ ] Cost vs. satisfaction scatter plot: Efficiency analysis
- [ ] Cost distribution: Histogram of food costs

#### 5.3 Meal Composition Visualizations
**Tasks:**
- [ ] Meal structure visualization: Main/Drink/Side proportions
- [ ] Daily meal plan timeline: Breakfast/Lunch/Dinner breakdown
- [ ] Food frequency chart: How often each food appears
- [ ] Meal composition ratios: Visual representation of ratios

#### 5.4 Sensitivity Analysis Visualizations
**Tasks:**
- [ ] Tornado diagram: Impact of parameter changes on objective
- [ ] Sensitivity curves: How objective changes with parameter variations
- [ ] Heatmap: Correlation between foods and nutrients
- [ ] Shadow price bar chart: Visualize constraint values
- [ ] Parameter perturbation plots: Show solution stability

#### 5.5 Solution Quality Visualizations
**Tasks:**
- [ ] Pareto frontier: Trade-off between cost and satisfaction
- [ ] Solution comparison charts: Compare different scenarios
- [ ] Efficiency metrics: Cost per calorie, satisfaction per dollar
- [ ] Nutrient density visualization: Nutrients per serving

---

## 🎯 Implementation Priority

### Phase 1: Data Expansion
1. Expand nutrients (macro + micro)
2. Replace foods with restaurant menu items
3. Collect real data manually

### Phase 2: Uncertainty Modeling
4. Add price uncertainty
5. Add satisfaction uncertainty
6. Add nutrient uncertainty
7. Update objective function

### Phase 3: Post-Optimization Analysis
8. Implement sensitivity analysis
9. Extract shadow prices
10. Create what-if scenarios

### Phase 4: Visualization
11. Create all visualizations
12. Generate reports
13. Interactive dashboard (optional)

---

## 📝 Notes
- Keep current model structure as base
- All changes should maintain model feasibility
- Test each addition incrementally
- Document all assumptions and data sources


# My Diet Planner - Optimization Model

## Overview
This notebook implements a diet planning optimization model that:
- Minimizes total cost while meeting nutritional requirements
- Considers satisfaction scores and meal composition rules

## Model Features
- **Nutritional Constraints**: Minimum/maximum requirements for Calories, Protein, Carbs, Fat
- **Budget Constraints**: Cost must be between $200-$400
- **Satisfaction Requirements**: Minimum total satisfaction threshold
- **Meal Composition Rules**: Complementary foods, mutual exclusions, required pairs

In [62]:
import numpy as np
import pandas as pd
import gamspy as gp
import gamspy.math as gpm
import sys

gp.set_options({'USE_PY_VAR_NAME': 'yes'})
m = gp.Container()

In [68]:
expanded_nutrients = [
    "Calories", "Protein", "Carbs", "Fat", "SaturatedFat", "TransFat", "Sugars",
    "Sodium", "Fiber", "VitaminA", "VitaminC", "VitaminD", "Calcium", "Iron", 
    "Potassium", "Cholesterol", "Caffeine"
]

restaurants_list = [
    "Chipotle", "Subway", "McDonalds", "PizzaHut", "TacoBell",
    "Starbucks", "Dunkin",
    "IceCream", "ColdStone"
]

restaurants_dict = {
    "Chipotle": ["Chicken_Burrito", "Steak_Bowl", "Veggie_Tacos", "Chicken_Salad"],
    "Subway": ["Turkey_Sandwich", "Veggie_Delite", "Chicken_Teriyaki", "Meatball_Marinara"],
    "McDonalds": ["Big_Mac", "Quarter_Pounder", "Chicken_Nuggets", "French_Fries"],
    "PizzaHut": ["Pepperoni_Pizza", "Cheese_Pizza", "Veggie_Pizza", "Breadsticks"],
    "TacoBell": ["Crunchwrap", "Taco", "Burrito", "Nachos"],
    "Starbucks": ["Latte", "Cappuccino", "Frappuccino", "Muffin", "Croissant"],
    "Dunkin": ["Coffee", "Donut", "Bagel"],
    "IceCream": ["Vanilla_Cone", "Chocolate_Sundae", "Strawberry_Scoop", "Cookie_Dough"],
    "ColdStone": ["IceCream_Cake", "Smoothie", "Milkshake"]
}

menu_items_data = [(r, item) for r, items in restaurants_dict.items() for item in items]
food_list = [f"{r}_{item}" for r, items in restaurants_dict.items() for item in items]


In [69]:
# Load nutrient data from CSV file
# CSV Format: Restaurant_MenuItem,Calories,Protein,Carbs,Fat,SaturatedFat,TransFat,Sugars,Sodium,Fiber,VitaminA,VitaminC,VitaminD,Calcium,Iron,Potassium,Cholesterol,Caffeine

csv_file = "nutrient_data_template.csv"
df = pd.read_csv(csv_file)

nutrient_values = {}
for _, row in df.iterrows():
    item = row['Restaurant_MenuItem']
    nutrient_values[item] = {
        "Calories": row['Calories'], "Protein": row['Protein'], "Carbs": row['Carbs'], "Fat": row['Fat'],
        "SaturatedFat": row['SaturatedFat'], "TransFat": row['TransFat'], "Sugars": row['Sugars'],
        "Sodium": row['Sodium'], "Fiber": row['Fiber'], "VitaminA": row['VitaminA'], "VitaminC": row['VitaminC'],
        "VitaminD": row['VitaminD'], "Calcium": row['Calcium'], "Iron": row['Iron'], "Potassium": row['Potassium'],
        "Cholesterol": row['Cholesterol'], "Caffeine": row['Caffeine']
    }

    
nutrient_data_expanded = [(food, nutrient, nutrient_values[food].get(nutrient, 0)) 
                          for food in food_list for nutrient in expanded_nutrients]



KeyError: 'PizzaHut_Pepperoni_Pizza'

In [64]:
from IPython.display import HTML, display
display(HTML('<iframe src="https://www.nutritionix.com/subway/nutrition-calculator" width="100%" height="800"></iframe>'))


In [3]:
import numpy as np
import pandas as pd
import gamspy as gp
import gamspy.math as gpm
import sys

gp.set_options({'USE_PY_VAR_NAME': 'yes'})
m = gp.Container()

In [ ]:
# Sets
# Create restaurants set
restaurants = gp.Set(m, name="restaurants", records=restaurants_list)

# Create menu items set with restaurants as domain
menu_items = gp.Set(m, name="menu_items", domain=[restaurants], records=menu_items_data)

# Foods are now restaurant menu items (e.g., "Chipotle_Chicken_Burrito")
foods = gp.Set(m, name="foods", records=food_list)

# Nutrients: Expanded to match standard Nutrition Facts label + additional nutrients
# Basic Macros: Calories, Protein, Carbs, Fat
# Fat Details: SaturatedFat, TransFat
# Carb Details: Sugars
# Vitamins: VitaminA, VitaminC, VitaminD
# Minerals: Calcium, Iron, Potassium, Sodium
# Other: Fiber, Cholesterol, Caffeine
nutrients = gp.Set(m, name="nutrients", records=expanded_nutrients)

# Parameters
# Price per serving (in dollars) - indexed over menu items
# Prices are estimates based on typical restaurant pricing
price_per_serving = gp.Parameter(m, name="price_per_serving", domain=[foods], 
                                  records=pd.Series({
                                      # Chipotle
                                      "Chipotle_Chicken_Burrito": 9.50,
                                      "Chipotle_Steak_Bowl": 10.50,
                                      "Chipotle_Veggie_Tacos": 8.00,
                                      "Chipotle_Chicken_Salad": 9.00,
                                      # Subway
                                      "Subway_Turkey_Sandwich": 6.50,
                                      "Subway_Veggie_Delite": 5.50,
                                      "Subway_Chicken_Teriyaki": 7.00,
                                      "Subway_Meatball_Marinara": 7.50,
                                      # McDonald's
                                      "McDonalds_Big_Mac": 5.99,
                                      "McDonalds_Quarter_Pounder": 5.49,
                                      "McDonalds_Chicken_Nuggets": 4.99,
                                      "McDonalds_French_Fries": 3.49,
                                      # Starbucks
                                      "Starbucks_Latte": 5.45,
                                      "Starbucks_Cappuccino": 4.95,
                                      "Starbucks_Frappuccino": 5.95,
                                      "Starbucks_Muffin": 3.25,
                                      "Starbucks_Croissant": 2.95,
                                      # Ice Cream
                                      "IceCream_Vanilla_Cone": 3.50,
                                      "IceCream_Chocolate_Sundae": 4.50,
                                      "IceCream_Strawberry_Scoop": 3.00,
                                      "IceCream_Cookie_Dough": 4.00
                                  }))

# Nutrient content per serving (indexed over foods and nutrients)
# Uses expanded nutrient data from previous cell (includes macros + micros)
nutrient_per_serving = gp.Parameter(m, name="nutrient_per_serving", domain=[foods, nutrients],
                                     records=nutrient_data_expanded)


In [ ]:
# Meal Composition: Categorize foods into meal components
# This creates structured meals (Main + Drink + Side)

# Food categories
food_categories = gp.Set(m, name="food_categories", records=["Main", "Drink", "Side"])

# Categorize each menu item
food_category = gp.Parameter(m, name="food_category", domain=[foods, food_categories],
                            records=[
                                # Main dishes (restaurant entrees)
                                ("Chipotle_Chicken_Burrito", "Main", 1), ("Chipotle_Steak_Bowl", "Main", 1),
                                ("Chipotle_Veggie_Tacos", "Main", 1), ("Chipotle_Chicken_Salad", "Main", 1),
                                ("Subway_Turkey_Sandwich", "Main", 1), ("Subway_Veggie_Delite", "Main", 1),
                                ("Subway_Chicken_Teriyaki", "Main", 1), ("Subway_Meatball_Marinara", "Main", 1),
                                ("McDonalds_Big_Mac", "Main", 1), ("McDonalds_Quarter_Pounder", "Main", 1),
                                ("McDonalds_Chicken_Nuggets", "Main", 1),
                                # Drinks
                                ("Starbucks_Latte", "Drink", 1), ("Starbucks_Cappuccino", "Drink", 1),
                                ("Starbucks_Frappuccino", "Drink", 1),
                                # Sides (snacks, desserts, sides)
                                ("McDonalds_French_Fries", "Side", 1),
                                ("Starbucks_Muffin", "Side", 1), ("Starbucks_Croissant", "Side", 1),
                                ("IceCream_Vanilla_Cone", "Side", 1), ("IceCream_Chocolate_Sundae", "Side", 1),
                                ("IceCream_Strawberry_Scoop", "Side", 1), ("IceCream_Cookie_Dough", "Side", 1)
                            ])

# Meal times (Breakfast, Lunch, Dinner)
meals = gp.Set(m, name="meals", records=["Breakfast", "Lunch", "Dinner"])

# Minimum servings per meal type (ensures balanced meals)
min_main_per_meal = gp.Parameter(m, name="min_main_per_meal", records=1.0)  # At least 1 main per meal
min_drink_per_meal = gp.Parameter(m, name="min_drink_per_meal", records=0.5)  # At least 0.5 drink per meal
min_side_per_meal = gp.Parameter(m, name="min_side_per_meal", records=1.0)  # At least 1 side per meal

print("Meal Composition Structure:")
print("  Main dishes: Chipotle, Subway, McDonald's entrees")
print("  Drinks: Starbucks beverages")
print("  Sides: Fries, pastries, ice cream")
print(f"  Meal times: {meals.toList()}")


In [ ]:
# Satisfaction Scores and Meal Composition Rules
# Satisfaction: How much you enjoy each menu item (scale 1-10, higher is better)
satisfaction = gp.Parameter(m, name="satisfaction", domain=[foods],
                           records=pd.Series({
                               # Chipotle
                               "Chipotle_Chicken_Burrito": 8.5,
                               "Chipotle_Steak_Bowl": 9.0,
                               "Chipotle_Veggie_Tacos": 7.5,
                               "Chipotle_Chicken_Salad": 7.0,
                               # Subway
                               "Subway_Turkey_Sandwich": 7.0,
                               "Subway_Veggie_Delite": 6.5,
                               "Subway_Chicken_Teriyaki": 7.5,
                               "Subway_Meatball_Marinara": 8.0,
                               # McDonald's
                               "McDonalds_Big_Mac": 8.0,
                               "McDonalds_Quarter_Pounder": 7.5,
                               "McDonalds_Chicken_Nuggets": 7.0,
                               "McDonalds_French_Fries": 8.5,
                               # Starbucks
                               "Starbucks_Latte": 8.0,
                               "Starbucks_Cappuccino": 7.5,
                               "Starbucks_Frappuccino": 9.0,
                               "Starbucks_Muffin": 6.5,
                               "Starbucks_Croissant": 7.0,
                               # Ice Cream
                               "IceCream_Vanilla_Cone": 8.5,
                               "IceCream_Chocolate_Sundae": 9.0,
                               "IceCream_Strawberry_Scoop": 8.0,
                               "IceCream_Cookie_Dough": 9.5
                           }))

# Minimum satisfaction threshold (constraint)
min_satisfaction = gp.Parameter(m, name="min_satisfaction", records=300.0)  # Minimum total satisfaction required

In [ ]:
# Update Nmin and Nmax with expanded nutrients
# This cell expands the nutrient constraints to include micro-nutrients

# Nmin: Minimum required levels (expanded)
Nmin_expanded = pd.Series({
    # Macros
    "Calories": 1800,   # Minimum calories per day
    "Protein": 70,      # Minimum grams of protein
    "Carbs": 400,       # Minimum grams of carbs
    "Fat": 200,         # Minimum grams of fat
    # Fat details
    "SaturatedFat": 0,  # Minimum grams saturated fat (no minimum requirement)
    "TransFat": 0,      # Minimum grams trans fat (no minimum requirement)
    # Carb details
    "Sugars": 0,        # Minimum grams sugars (no minimum requirement)
    # Micros
    "Sodium": 500,      # Minimum mg sodium (very low minimum)
    "Fiber": 25,        # Minimum grams of fiber
    "VitaminA": 700,    # Minimum mcg RAE vitamin A
    "VitaminC": 60,     # Minimum mg vitamin C
    "VitaminD": 15,     # Minimum mcg vitamin D (RDA)
    "Calcium": 1000,     # Minimum mg calcium
    "Iron": 8,          # Minimum mg iron
    "Potassium": 2600,  # Minimum mg potassium (RDA for adults)
    "Cholesterol": 0,   # Minimum mg cholesterol (no minimum requirement)
    "Caffeine": 0       # Minimum mg caffeine (no minimum requirement)
})

# Nmax: Maximum allowable levels (expanded)
Nmax_expanded = pd.Series({
    # Macros
    "Calories": 25000,   # Maximum calories per day
    "Protein": 1500,     # Maximum grams of protein
    "Carbs": 5000,       # Maximum grams of carbs
    "Fat": 30000,        # Maximum grams of fat
    # Fat details
    "SaturatedFat": 200, # Maximum grams saturated fat (FDA recommends <20g, but set higher for constraint)
    "TransFat": 2,       # Maximum grams trans fat (FDA recommends as low as possible, <2g)
    # Carb details
    "Sugars": 100,       # Maximum grams added sugars (FDA recommends <50g, but includes natural sugars)
    # Micros
    "Sodium": 2300,      # Maximum mg sodium (FDA daily limit)
    "Fiber": 100,        # Maximum grams of fiber (reasonable upper bound)
    "VitaminA": 3000,    # Maximum mcg RAE vitamin A (UL)
    "VitaminC": 2000,    # Maximum mg vitamin C (UL)
    "VitaminD": 100,     # Maximum mcg vitamin D (UL)
    "Calcium": 2500,     # Maximum mg calcium (UL)
    "Iron": 45,          # Maximum mg iron (UL)
    "Potassium": 5000,   # Maximum mg potassium (reasonable upper bound, no UL established)
    "Cholesterol": 300,  # Maximum mg cholesterol (FDA daily limit)
    "Caffeine": 400      # Maximum mg caffeine (FDA safe limit for adults)
})

# Update the parameters
Nmin = gp.Parameter(m, name="Nmin", domain=[nutrients], records=Nmin_expanded)
Nmax = gp.Parameter(m, name="Nmax", domain=[nutrients], records=Nmax_expanded)

print("✅ Nutrient constraints updated with expanded nutrients:")
print(f"   Basic Macros: Calories, Protein, Carbs, Fat")
print(f"   Fat Details: SaturatedFat, TransFat")
print(f"   Carb Details: Sugars")
print(f"   Vitamins: VitaminA, VitaminC, VitaminD")
print(f"   Minerals: Calcium, Iron, Potassium, Sodium")
print(f"   Other: Fiber, Cholesterol, Caffeine")


In [5]:
# Weight for multi-objective: balance between cost (w1) and satisfaction (w2)
# w1*cost - w2*satisfaction (minimize cost, maximize satisfaction)
w_cost = gp.Parameter(m, name="w_cost", records=1.0)      # Weight for cost
w_satisfaction = gp.Parameter(m, name="w_satisfaction", records=0.5)  # Weight for satisfaction

# Create sets for meal composition rules
# Rule 1: Complementary foods (pairs that go well together)
complementary_list = [
    ("Rice", "Chicken"),      # If eating rice, should have chicken (protein)
    ("Bread", "Banana"),     # If bread, pair with fruit
    ("Oats", "Banana")       # Oats with banana
]

# Rule 2: Mutual exclusion (cannot eat both together)
mutual_excl_list = [
    ("Chipotle", "McDonalds"),  # Don't eat both fast food restaurants
    ("Chipotle", "Dominoes"),   # Don't eat multiple restaurant meals
    ("McDonalds", "Dominoes")   # Choose one restaurant option
]

In [6]:
# Food serving constraints: Fmini and Fmaxi
# Fmini = minimum number of required servings of food i, ∀i∈F
Fmin = gp.Parameter(m, name="Fmin", domain=[foods], 
                    records=pd.Series({
                        "Chipotle": 2,      # Minimum servings (can be 0 if food is optional)
                        "McDonalds": 2,
                        "Dominoes": 2,
                        "Chicken": 4,
                        "Rice": 8,
                        "Milk": 10,
                        "Bread": 10,
                        "Oats": 10,
                        "Banana": 10,
                        "Apple": 10,
                        "Grapes": 10,
                        "Orange": 10,
                        "Pineapple": 5
                    }))

# Fmaxi = maximum allowable number of servings of food i, ∀i∈F
Fmax = gp.Parameter(m, name="Fmax", domain=[foods],
                    records=pd.Series({
                        "Chipotle": 14,     # Maximum servings per day
                        "McDonalds": 14,
                        "Dominoes": 14,
                        "Chicken": 14,
                        "Rice": 14,
                        "Milk": 14,
                        "Bread": 14,
                        "Oats": 14,
                        "Banana": 14,
                        "Apple": 14,
                        "Grapes": 14,
                        "Orange": 14,
                        "Pineapple": 14
                    }))

# Nutrient level constraints: Nminj and Nmaxj
# Nminj = minimum required level of nutrient j, ∀j∈N
Nmin = gp.Parameter(m, name="Nmin", domain=[nutrients],
                    records=pd.Series({
                        "Calories": 1800,   # Minimum calories per day
                        "Protein": 70,      # Minimum grams of protein
                        "Carbs": 400,       # Minimum grams of carbs
                        "Fat": 200           # Minimum grams of fat
                    }))

# Nmaxj = maximum allowable level of nutrient j, ∀j∈N
Nmax = gp.Parameter(m, name="Nmax", domain=[nutrients],
                    records=pd.Series({
                        "Calories": 25000,   # Maximum calories per day
                        "Protein": 1500,     # Maximum grams of protein
                        "Carbs": 5000,       # Maximum grams of carbs
                        "Fat": 30000           # Maximum grams of fat
                    }))

In [7]:
# Variables
# xi = number of servings of food i to purchase/consume, ∀i∈F
x = gp.Variable(m, name="x", domain=[foods], type="positive", description="number of servings of food i")

# Set bounds using Fmin and Fmax parameters
x.lo[foods] = Fmin[foods]  # Lower bound: minimum servings
x.up[foods] = Fmax[foods]  # Upper bound: maximum servings

In [8]:
# Equations (Constraints)

# Constraint Set 1: For each nutrient j∈N, at least meet the minimum required level
# ∑(i∈F) aij*xi ≥ Nminj, ∀j∈N
nutrient_min = gp.Equation(m, name="nutrient_min", domain=[nutrients], description="minimum nutrient requirements")
nutrient_min[nutrients] = gp.Sum(foods, nutrient_per_serving[foods, nutrients] * x[foods]) >= Nmin[nutrients]

# Constraint Set 2: For each nutrient j∈N, do not exceed the maximum allowable level
# ∑(i∈F) aij*xi ≤ Nmaxj, ∀j∈N
nutrient_max = gp.Equation(m, name="nutrient_max", domain=[nutrients], description="maximum nutrient limits")
nutrient_max[nutrients] = gp.Sum(foods, nutrient_per_serving[foods, nutrients] * x[foods]) <= Nmax[nutrients]

# Constraint: Total cost (budget) must be between $150 and $200
# 150 ≤ ∑(i∈F) ci*xi ≤ 200
cost_min = gp.Equation(m, name="cost_min", description="minimum budget constraint ($200)")
cost_min[:] = gp.Sum(foods, price_per_serving[foods] * x[foods]) >= 200

cost_max = gp.Equation(m, name="cost_max", description="maximum budget constraint ($400)")
cost_max[:] = gp.Sum(foods, price_per_serving[foods] * x[foods]) <= 400
# Note: Constraint Set 3 (xi ≥ Fmini) and Constraint Set 4 (xi ≤ Fmaxi) 
# are already handled by the variable bounds set in Cell 4

# Constraint: Total satisfaction must be above minimum threshold
# ∑(i∈F) satisfaction[i] * x[i] ≥ min_satisfaction
satisfaction_constraint = gp.Equation(m, name="satisfaction_constraint", description="Minimum satisfaction requirement")
satisfaction_constraint[:] = gp.Sum(foods, satisfaction[foods] * x[foods]) >= min_satisfaction

In [ ]:
# Meal Composition Constraints: Ensure balanced meals
# Each meal should have: Main + Drink + Side

# Calculate total servings per category using explicit sums
total_main_servings = (x["Chipotle"] + x["McDonalds"] + x["Dominoes"] + 
                       x["Chicken"] + x["Rice"])
total_drink_servings = x["Milk"]
total_side_servings = (x["Bread"] + x["Oats"] + x["Banana"] + x["Apple"] + 
                       x["Grapes"] + x["Orange"] + x["Pineapple"])

# Constraint: Ensure balanced meal composition
# Ratio constraints ensure proper meal structure

# Minimum ratios (ensures balanced meals)
# For every main dish, have proportional drinks and sides
meal_balance_main_drink = gp.Equation(m, name="meal_balance_main_drink", 
                                      description="Main to Drink ratio (min)")
meal_balance_main_drink[:] = total_drink_servings >= 0.3 * total_main_servings

meal_balance_main_side = gp.Equation(m, name="meal_balance_main_side", 
                                     description="Main to Side ratio (min)")
meal_balance_main_side[:] = total_side_servings >= 0.8 * total_main_servings

# Maximum ratios (prevents overconsumption of one category)
meal_balance_max_drink = gp.Equation(m, name="meal_balance_max_drink", 
                                     description="Max Drink to Main ratio")
meal_balance_max_drink[:] = total_drink_servings <= 2.0 * total_main_servings

meal_balance_max_side = gp.Equation(m, name="meal_balance_max_side", 
                                    description="Max Side to Main ratio")
meal_balance_max_side[:] = total_side_servings <= 3.0 * total_main_servings

print("Meal composition constraints added:")
print("  - Drink servings >= 30% of Main servings")
print("  - Side servings >= 80% of Main servings")
print("  - Drink servings <= 200% of Main servings")
print("  - Side servings <= 300% of Main servings")


In [ ]:
# Meal Composition Rules Constraints

# Rule 1: Complementary Foods - Pair main dishes with drinks
# If you get a main dish, encourage getting a drink
total_main_items = (x["Chipotle_Chicken_Burrito"] + x["Chipotle_Steak_Bowl"] + x["Chipotle_Veggie_Tacos"] + 
                   x["Chipotle_Chicken_Salad"] + x["Subway_Turkey_Sandwich"] + x["Subway_Veggie_Delite"] +
                   x["Subway_Chicken_Teriyaki"] + x["Subway_Meatball_Marinara"] + x["McDonalds_Big_Mac"] +
                   x["McDonalds_Quarter_Pounder"] + x["McDonalds_Chicken_Nuggets"])

total_drink_items = (x["Starbucks_Latte"] + x["Starbucks_Cappuccino"] + x["Starbucks_Frappuccino"])

comp_main_drink = gp.Equation(m, name="comp_main_drink", description="Main dishes pair with drinks")
comp_main_drink[:] = total_drink_items >= 0.3 * total_main_items

# Rule 2: Mutual Exclusion - Limit servings from different fast food restaurants
# Don't eat too many items from different restaurants in one day

# Chipotle and McDonald's mutual exclusion (limit combined servings)
chipotle_total = (x["Chipotle_Chicken_Burrito"] + x["Chipotle_Steak_Bowl"] + 
                 x["Chipotle_Veggie_Tacos"] + x["Chipotle_Chicken_Salad"])
mcdonalds_total = (x["McDonalds_Big_Mac"] + x["McDonalds_Quarter_Pounder"] + 
                   x["McDonalds_Chicken_Nuggets"] + x["McDonalds_French_Fries"])
subway_total = (x["Subway_Turkey_Sandwich"] + x["Subway_Veggie_Delite"] +
                x["Subway_Chicken_Teriyaki"] + x["Subway_Meatball_Marinara"])

mutex_fastfood = gp.Equation(m, name="mutex_fastfood", description="Limit fast food restaurant variety")
mutex_fastfood[:] = chipotle_total + mcdonalds_total + subway_total <= 8

# Rule 3: Pair sides with main dishes
total_side_items = (x["McDonalds_French_Fries"] + x["Starbucks_Muffin"] + x["Starbucks_Croissant"] +
                    x["IceCream_Vanilla_Cone"] + x["IceCream_Chocolate_Sundae"] + 
                    x["IceCream_Strawberry_Scoop"] + x["IceCream_Cookie_Dough"])

comp_main_side = gp.Equation(m, name="comp_main_side", description="Main dishes pair with sides")
comp_main_side[:] = total_side_items >= 0.5 * total_main_items

In [10]:
# Objective Function: Multi-objective optimization
# Minimize: w_cost * cost - w_satisfaction * satisfaction
# This balances minimizing cost while maximizing satisfaction

total_cost = gp.Sum(foods, price_per_serving[foods] * x[foods])
total_satisfaction = gp.Sum(foods, satisfaction[foods] * x[foods])

# Objective: minimize weighted combination
# Lower values = better (less cost, more satisfaction)
# Objective Function: Minimize total cost
# Satisfaction is now a constraint (must be >= threshold), not part of objective
obj_expr = gp.Sum(foods, price_per_serving[foods] * x[foods])

In [ ]:
# Display Meal Composition Breakdown
print("\n" + "="*70)
print("MEAL COMPOSITION BREAKDOWN")
print("="*70)

# Get solution values
food_col = x.records.columns[0]
x_values = {}
for idx, row in x.records.iterrows():
    food_name = row[food_col]
    x_values[food_name] = row['level']

# Calculate category totals
main_total = sum(x_values.get(f, 0) for f in ["Chipotle", "McDonalds", "Dominoes", "Chicken", "Rice"])
drink_total = x_values.get("Milk", 0)
side_total = sum(x_values.get(f, 0) for f in ["Bread", "Oats", "Banana", "Apple", "Grapes", "Orange", "Pineapple"])

print(f"\n📊 Total Servings by Category:")
print(f"  Main Dishes: {main_total:.1f} servings")
print(f"  Drinks:      {drink_total:.1f} servings")
print(f"  Sides:       {side_total:.1f} servings")

print(f"\n🍽️  Meal Composition Ratios:")
if main_total > 0:
    print(f"  Drink/Main:  {drink_total/main_total:.2f} (target: 0.3-2.0)")
    print(f"  Side/Main:   {side_total/main_total:.2f} (target: 0.8-3.0)")

print(f"\n📋 Detailed Breakdown:")
print(f"\n  MAIN DISHES:")
for f in ["Chipotle", "McDonalds", "Dominoes", "Chicken", "Rice"]:
    if x_values.get(f, 0) > 0:
        print(f"    • {f}: {x_values.get(f, 0):.1f} servings")

print(f"\n  DRINKS:")
if x_values.get("Milk", 0) > 0:
    print(f"    • Milk: {x_values.get('Milk', 0):.1f} servings")

print(f"\n  SIDES:")
for f in ["Bread", "Oats", "Banana", "Apple", "Grapes", "Orange", "Pineapple"]:
    if x_values.get(f, 0) > 0:
        print(f"    • {f}: {x_values.get(f, 0):.1f} servings")


In [11]:


# Create and solve the model
diet_plan = gp.Model(
    m,
    equations=m.getEquations(),
    problem=gp.Problem.LP,
    sense=gp.Sense.MIN,  # Minimize total cost
    objective=obj_expr,
    name="diet_plan",
)

# Solve the model
diet_plan.solve(output=sys.stdout)



--- Job _nS_tAUdkQLOp4Fx_VwpQOg.gms Start 12/10/25 00:33:30 52.1.0 4f802a74 WEX-WEI x86 64bit/MS Windows
--- Applying:
    C:\Users\Shashwat\Desktop\CS 524 Introduction to optimization\.venv\Lib\site-packages\gamspy_base\gmsprmNT.txt
--- GAMS Parameters defined
    LP cplex
    Input C:\Users\Shashwat\AppData\Local\Temp\tmpoiejb_yz\_nS_tAUdkQLOp4Fx_VwpQOg.gms
    Output C:\Users\Shashwat\AppData\Local\Temp\tmpoiejb_yz\_nS_tAUdkQLOp4Fx_VwpQOg.lst
    ScrDir C:\Users\Shashwat\AppData\Local\Temp\tmpoiejb_yz\tmpdy1c8l7y\
    SysDir "C:\Users\Shashwat\Desktop\CS 524 Introduction to optimization\.venv\Lib\site-packages\gamspy_base\"
    LogOption 3
    Trace C:\Users\Shashwat\AppData\Local\Temp\tmpoiejb_yz\_nS_tAUdkQLOp4Fx_VwpQOg.txt
    License C:\Users\Shashwat\Documents\GAMSPy\gamspy_license.txt
    OptFile 0
    OptDir C:\Users\Shashwat\AppData\Local\Temp\tmpoiejb_yz\
    LimRow 0
    LimCol 0
    TraceOpt 3
    GDX C:\Users\Shashwat\AppData\Local\Temp\tmpoiejb_yz\_nS_tAUdkQLOp4Fx_VwpQOg

,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,Normal,OptimalGlobal,200.0,19,14,LP,CPLEX,0.016


In [12]:
# Display results
print("\n=== OPTIMIZATION RESULTS ===")
print(f"Objective Function Value (Total Cost): ${diet_plan.objective_value:.2f}")
print(f"\nSolution Status: {diet_plan.status}")
print(f"Solver Status: {diet_plan.solve_status}")

print("\n=== FOOD SERVINGS ===")
print(x.records)

print("\n=== NUTRIENT TOTALS ===")
# Calculate total nutrients from solution by extracting numeric values

# Get the food column name from x.records (first column)
food_col = x.records.columns[0]  # Should be 'foods' or similar

# Create a dictionary mapping food names to their serving amounts from solution
x_values = {}
for idx, row in x.records.iterrows():
    food_name = row[food_col]
    x_values[food_name] = row['level']

# Calculate nutrient totals
nutrient_totals = {}
for n in nutrients.toList():
    total = 0.0
    # Get nutrient values from parameter records
    for idx, row in nutrient_per_serving.records.iterrows():
        # Check if this row is for the current nutrient
        nutrient_name = row[nutrients.name] if nutrients.name in row.index else row.iloc[1]  # Adjust based on your structure
        food_name = row[foods.name] if foods.name in row.index else row.iloc[0]
        
        if nutrient_name == n and food_name in x_values:
            nutrient_amt = row['value']
            servings = x_values[food_name]
            total += nutrient_amt * servings
    
    nutrient_totals[n] = total
    print(f"{n}: {total:.2f} (Min: {Nmin[n]}, Max: {Nmax[n]})")


=== OPTIMIZATION RESULTS ===
Objective Function Value (Total Cost): $200.00

Solution Status: ModelStatus.OptimalGlobal
Solver Status: SolveStatus.NormalCompletion

=== FOOD SERVINGS ===
        foods  level  marginal  lower  upper  scale
0    Chipotle    2.0      -0.0    2.0   14.0    1.0
1   McDonalds    2.0      -0.0    2.0   14.0    1.0
2    Dominoes    2.0      -0.0    2.0   14.0    1.0
3     Chicken    7.3       0.0    4.0   14.0    1.0
4        Rice    8.0      -0.0    8.0   14.0    1.0
5        Milk   10.0      -0.0   10.0   14.0    1.0
6       Bread   10.0      -0.0   10.0   14.0    1.0
7        Oats   10.0      -0.0   10.0   14.0    1.0
8      Banana   10.0      -0.0   10.0   14.0    1.0
9       Apple   10.0      -0.0   10.0   14.0    1.0
10     Grapes   10.0      -0.0   10.0   14.0    1.0
11     Orange   10.0      -0.0   10.0   14.0    1.0
12  Pineapple    5.0      -0.0    5.0   14.0    1.0

=== NUTRIENT TOTALS ===
Calories: 14125.00 (Min: ImplicitParameter(parent=Parameter